# Income Classification Using PySpark

## Project Overview

This portfolio project develops a machine learning classification workflow to predict whether an individual's annual income is above or below $50,000 using the UCI Adult Census Income dataset.

The project combines **technical data-science skills** with a **business-oriented interpretation** of the results. It demonstrates data acquisition, data quality assessment, exploratory data analysis, feature engineering, scalable preprocessing with PySpark, model development, evaluation, feature importance and responsible business interpretation.

### Business Question

**Can demographic, educational, employment and financial characteristics be used to distinguish between individuals earning more than $50,000 and those earning $50,000 or less?**

### Portfolio Skills Demonstrated

- Python and PySpark
- Data cleaning and quality assessment
- Exploratory Data Analysis (EDA)
- Feature engineering
- Categorical encoding
- Machine learning pipelines
- Decision Trees and Random Forests
- Model evaluation
- Model comparison
- Feature importance
- Business interpretation
- Responsible AI and fairness considerations

## 1. Dataset and Data Source

The project uses the **Adult (Census Income)** dataset from the UCI Machine Learning Repository.

The dataset contains demographic, educational, employment and financial characteristics. The target variable indicates whether annual income is:

- `<=50K`
- `>50K`

**Source:** UCI Machine Learning Repository  
**Dataset:** Adult  
**DOI:** https://doi.org/10.24432/C5XW20

The raw dataset is intentionally not included in this GitHub repository. See `data/README.md` for instructions on obtaining it.

In [ ]:
# Install PySpark in Google Colab
!pip install -q pyspark

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("Libraries imported successfully.")

In [ ]:
spark = (
    SparkSession.builder
    .appName("IncomeClassification")
    .getOrCreate()
)

print("Spark version:", spark.version)

## 2. Data Acquisition

The notebook expects the UCI Adult dataset to be available as `adult.zip` in the Colab working directory.

If the ZIP file has already been uploaded, the following cell extracts it. Otherwise, upload `adult.zip` through the Colab Files panel before running the cell.

In [ ]:
# Extract the dataset if the ZIP file is present
if os.path.exists("adult.zip"):
    with zipfile.ZipFile("adult.zip", "r") as z:
        z.extractall(".")
    print("Dataset extracted successfully.")
else:
    print("adult.zip was not found.")
    print("Upload adult.zip to Colab, then run this cell again.")

In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

df = spark.read.csv(
    "adult.data",
    header=False,
    inferSchema=True
).toDF(*columns)

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.show(5, truncate=False)

## 3. Data Dictionary

| Variable | Description | Type |
|---|---|---|
| age | Age of the individual | Numerical |
| workclass | Employment sector | Categorical |
| fnlwgt | Final sampling weight | Numerical |
| education | Highest education level | Categorical |
| education_num | Numerical representation of education | Numerical |
| marital_status | Marital status | Categorical |
| occupation | Occupation category | Categorical |
| relationship | Household relationship | Categorical |
| race | Race category | Categorical |
| sex | Sex category | Categorical |
| capital_gain | Capital gain | Numerical |
| capital_loss | Capital loss | Numerical |
| hours_per_week | Hours worked per week | Numerical |
| native_country | Country of origin | Categorical |
| income | Income classification target | Target |

`fnlwgt` is retained in the dataset for completeness but is not used as a predictive feature in the final modelling pipeline.

## 4. Data Quality Assessment and Cleaning

The raw dataset uses `?` to represent missing categorical values. Instead of dropping those observations, the project replaces these values with `Unknown` so that potentially useful records are retained.

Exact duplicate records are also removed.

This approach demonstrates a deliberate data-quality decision rather than automatically deleting incomplete observations.

In [ ]:
# Inspect missing-value markers
for column in ["workclass", "occupation", "native_country"]:
    missing_count = df.filter(F.trim(F.col(column)) == "?").count()
    print(f"{column}: {missing_count}")

In [ ]:
categorical_columns = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

clean_df = df

for column in categorical_columns:
    clean_df = clean_df.withColumn(
        column,
        F.when(
            F.trim(F.col(column)) == "?",
            "Unknown"
        ).otherwise(F.trim(F.col(column)))
    )

clean_df = clean_df.withColumn("income", F.trim(F.col("income")))

# Create binary target
clean_df = clean_df.withColumn(
    "label",
    F.when(F.col("income") == ">50K", 1.0).otherwise(0.0)
)

# Remove exact duplicates
before = clean_df.count()
clean_df = clean_df.dropDuplicates()
after = clean_df.count()

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)
print("Duplicates removed:", before - after)

In [ ]:
clean_df.groupBy("income", "label").count().orderBy("label").show()

print("Final rows:", clean_df.count())
print("Final columns:", len(clean_df.columns))

## 5. Exploratory Data Analysis

EDA is used to understand the target distribution and investigate relationships between income classification and demographic, educational, employment and financial characteristics.

The purpose is not to prove causation. The analysis identifies patterns that may be useful for prediction and business interpretation.

In [ ]:
income_counts = (
    clean_df.groupBy("income").count().orderBy("income").toPandas()
)

plt.figure(figsize=(7, 5))
sns.barplot(data=income_counts, x="income", y="count")
plt.title("Income Class Distribution")
plt.xlabel("Income Class")
plt.ylabel("Number of Individuals")
plt.tight_layout()
plt.show()

income_counts

In [ ]:
age_income = clean_df.select("age", "income").toPandas()

plt.figure(figsize=(9, 5))
sns.boxplot(data=age_income, x="income", y="age")
plt.title("Age Distribution by Income Class")
plt.xlabel("Income Class")
plt.ylabel("Age")
plt.tight_layout()
plt.show()

In [ ]:
education_income = (
    clean_df
    .groupBy("education", "income")
    .count()
)

education_totals = (
    education_income
    .groupBy("education")
    .agg(F.sum("count").alias("total"))
)

education_pct = (
    education_income
    .join(education_totals, "education")
    .withColumn(
        "percentage",
        F.round(F.col("count") / F.col("total") * 100, 2)
    )
)

education_above = (
    education_pct
    .filter(F.col("income") == ">50K")
    .orderBy(F.desc("percentage"))
    .toPandas()
)

plt.figure(figsize=(10, 7))
sns.barplot(data=education_above, x="percentage", y="education")
plt.title("Percentage Earning Above $50K by Education Level")
plt.xlabel("Percentage Above $50K")
plt.ylabel("Education")
plt.tight_layout()
plt.show()

In [ ]:
hours_income = (
    clean_df
    .groupBy("income")
    .agg(F.avg("hours_per_week").alias("average_hours"))
    .toPandas()
)

plt.figure(figsize=(7, 5))
sns.barplot(data=hours_income, x="income", y="average_hours")
plt.title("Average Working Hours by Income Class")
plt.xlabel("Income Class")
plt.ylabel("Average Hours per Week")
plt.tight_layout()
plt.show()

hours_income

In [ ]:
occupation_income = clean_df.groupBy("occupation", "income").count()

occupation_totals = (
    occupation_income
    .groupBy("occupation")
    .agg(F.sum("count").alias("total"))
)

occupation_above = (
    occupation_income
    .join(occupation_totals, "occupation")
    .withColumn(
        "percentage",
        F.col("count") / F.col("total") * 100
    )
    .filter(F.col("income") == ">50K")
    .orderBy(F.desc("percentage"))
    .toPandas()
)

plt.figure(figsize=(10, 7))
sns.barplot(data=occupation_above, x="percentage", y="occupation")
plt.title("Percentage Earning Above $50K by Occupation")
plt.xlabel("Percentage Above $50K")
plt.ylabel("Occupation")
plt.tight_layout()
plt.show()

In [ ]:
sex_income = clean_df.groupBy("sex", "income").count()

sex_totals = sex_income.groupBy("sex").agg(F.sum("count").alias("total"))

sex_above = (
    sex_income
    .join(sex_totals, "sex")
    .withColumn(
        "percentage",
        F.col("count") / F.col("total") * 100
    )
    .filter(F.col("income") == ">50K")
    .orderBy(F.desc("percentage"))
    .toPandas()
)

plt.figure(figsize=(7, 5))
sns.barplot(data=sex_above, x="sex", y="percentage")
plt.title("Percentage Earning Above $50K by Sex")
plt.xlabel("Sex")
plt.ylabel("Percentage Above $50K")
plt.tight_layout()
plt.show()

sex_above

In [ ]:
workclass_income = clean_df.groupBy("workclass", "income").count()
workclass_totals = workclass_income.groupBy("workclass").agg(F.sum("count").alias("total"))

workclass_above = (
    workclass_income
    .join(workclass_totals, "workclass")
    .withColumn(
        "percentage",
        F.col("count") / F.col("total") * 100
    )
    .filter(F.col("income") == ">50K")
    .orderBy(F.desc("percentage"))
    .toPandas()
)

plt.figure(figsize=(10, 6))
sns.barplot(data=workclass_above, x="percentage", y="workclass")
plt.title("Percentage Earning Above $50K by Workclass")
plt.xlabel("Percentage Above $50K")
plt.ylabel("Workclass")
plt.tight_layout()
plt.show()

In [ ]:
capital_summary = (
    clean_df
    .groupBy("income")
    .agg(
        F.avg("capital_gain").alias("mean_capital_gain"),
        F.expr("percentile_approx(capital_gain, 0.5)").alias("median_capital_gain"),
        F.avg("capital_loss").alias("mean_capital_loss"),
        F.expr("percentile_approx(capital_loss, 0.5)").alias("median_capital_loss")
    )
)

capital_summary.show()

In [ ]:
numeric_columns = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

numeric_pd = clean_df.select(numeric_columns).toPandas()
correlation_matrix = numeric_pd.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

### EDA Findings

The exploratory analysis provides several modelling insights:

- The target variable is imbalanced, with fewer observations in the `>50K` class.
- Age, education, occupation, workclass and working hours show different distributions across income classes.
- Education and employment characteristics show meaningful variation in the proportion of observations earning above $50K.
- Capital gain and capital loss are highly skewed financial variables and differ across income classes.
- Demographic variables show differences in income proportions and therefore require careful interpretation and fairness considerations.

These findings motivate the use of a classification model, while also highlighting why model performance should be evaluated using more than accuracy alone.

## 6. Feature Engineering and Data Preparation

The modelling pipeline uses:

**Numerical features**
- age
- education_num
- capital_gain
- capital_loss
- hours_per_week

**Categorical features**
- workclass
- education
- marital_status
- occupation
- relationship
- race
- sex
- native_country

`fnlwgt` is excluded from the predictive feature set because it is a survey sampling weight rather than a direct individual characteristic.

In [ ]:
numerical_features = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

categorical_features = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

feature_columns = numerical_features + categorical_features

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

In [ ]:
train_df, test_df = clean_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

print("Training class distribution:")
train_df.groupBy("label").count().orderBy("label").show()

print("Testing class distribution:")
test_df.groupBy("label").count().orderBy("label").show()

In [ ]:
indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=f"{column}_index",
        handleInvalid="keep"
    )
    for column in categorical_features
]

encoder = OneHotEncoder(
    inputCols=[f"{column}_index" for column in categorical_features],
    outputCols=[f"{column}_encoded" for column in categorical_features],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=(
        numerical_features +
        [f"{column}_encoded" for column in categorical_features]
    ),
    outputCol="features"
)

preprocessing_pipeline = Pipeline(
    stages=indexers + [encoder, assembler]
)

preprocessing_model = preprocessing_pipeline.fit(train_df)

train_prepared = preprocessing_model.transform(train_df)
test_prepared = preprocessing_model.transform(test_df)

train_prepared.select("label", "features").show(5, truncate=False)

## 7. Baseline Model — Decision Tree

A Decision Tree is used as the baseline classifier because it is relatively interpretable and can model nonlinear relationships between features and income classification.

The baseline uses a maximum tree depth of 5.

In [ ]:
dt = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    seed=42
)

dt_model = dt.fit(train_prepared)
predictions = dt_model.transform(test_prepared)

print("Baseline Decision Tree trained.")

In [ ]:
def evaluate_model(predictions):
    tp = predictions.filter(
        (F.col("label") == 1) & (F.col("prediction") == 1)
    ).count()

    tn = predictions.filter(
        (F.col("label") == 0) & (F.col("prediction") == 0)
    ).count()

    fp = predictions.filter(
        (F.col("label") == 0) & (F.col("prediction") == 1)
    ).count()

    fn = predictions.filter(
        (F.col("label") == 1) & (F.col("prediction") == 0)
    ).count()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )

    evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    auc = evaluator.evaluate(predictions)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "AUC": auc
    }

baseline_results = evaluate_model(predictions)
pd.DataFrame([baseline_results], index=["Baseline Decision Tree"]).round(4)

In [ ]:
cm_rows = (
    predictions
    .groupBy("label", "prediction")
    .count()
    .collect()
)

cm = np.zeros((2, 2), dtype=int)

for row in cm_rows:
    cm[int(row["label"]), int(row["prediction"])] = row["count"]

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["<=50K", ">50K"],
    yticklabels=["<=50K", ">50K"]
)
plt.xlabel("Predicted Income")
plt.ylabel("Actual Income")
plt.title("Baseline Decision Tree — Confusion Matrix")
plt.tight_layout()
plt.show()

print(cm)

## 8. Model Improvement — Deeper Decision Tree

The baseline model is intentionally simple. A second Decision Tree with a maximum depth of 8 is evaluated to determine whether additional model complexity improves the balance between precision and recall.

In [ ]:
dt_deeper = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=8,
    seed=42
)

dt_deeper_model = dt_deeper.fit(train_prepared)
deeper_predictions = dt_deeper_model.transform(test_prepared)

deeper_results = evaluate_model(deeper_predictions)

pd.DataFrame([deeper_results], index=["Deeper Decision Tree"]).round(4)

## 9. Random Forest Model

A Random Forest is added as a second modelling approach. It combines multiple decision trees and can reduce the variance associated with an individual tree.

The model is evaluated using the same test set and metrics as the Decision Tree models.

In [ ]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=150,
    maxDepth=8,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_prepared)
rf_predictions = rf_model.transform(test_prepared)

rf_results = evaluate_model(rf_predictions)

pd.DataFrame([rf_results], index=["Random Forest"]).round(4)

## 10. Model Comparison

The models are compared using accuracy, precision, recall, F1-score and AUC.

Because the dataset is imbalanced, **F1-score, recall and AUC are given substantial consideration rather than selecting a model solely on accuracy**.

In [ ]:
model_comparison = pd.DataFrame({
    "Baseline Decision Tree": baseline_results,
    "Deeper Decision Tree": deeper_results,
    "Random Forest": rf_results
}).T

model_comparison.round(4)

In [ ]:
model_comparison.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model using F1-score, with AUC as the secondary criterion.
ranking = model_comparison.sort_values(
    by=["F1-Score", "AUC"],
    ascending=False
)

best_model_name = ranking.index[0]

print("Recommended final model:", best_model_name)
print()
print(ranking.round(4))

## 11. Final Model Evaluation

The selected model is evaluated in greater detail using its confusion matrix and final performance metrics.

The final model is selected based on the balance of F1-score and AUC rather than accuracy alone. This reflects the project's focus on identifying both income classes effectively.

In [ ]:
model_objects = {
    "Baseline Decision Tree": (dt_model, predictions),
    "Deeper Decision Tree": (dt_deeper_model, deeper_predictions),
    "Random Forest": (rf_model, rf_predictions)
}

final_model, final_predictions = model_objects[best_model_name]

final_metrics = evaluate_model(final_predictions)

final_results = pd.DataFrame(
    [final_metrics],
    index=[best_model_name]
).round(4)

final_results

In [ ]:
final_cm_rows = (
    final_predictions
    .groupBy("label", "prediction")
    .count()
    .collect()
)

final_cm = np.zeros((2, 2), dtype=int)

for row in final_cm_rows:
    final_cm[int(row["label"]), int(row["prediction"])] = row["count"]

plt.figure(figsize=(6, 5))
sns.heatmap(
    final_cm,
    annot=True,
    fmt="d",
    xticklabels=["<=50K", ">50K"],
    yticklabels=["<=50K", ">50K"]
)
plt.xlabel("Predicted Income")
plt.ylabel("Actual Income")
plt.title(f"{best_model_name} — Final Confusion Matrix")
plt.tight_layout()
plt.show()

print("Confusion matrix:")
print(final_cm)

## 12. Feature Importance

Feature importance is used to identify which variables the final tree-based model relies on most strongly.

For one-hot encoded categorical variables, individual encoded categories are mapped back to readable feature names. Feature importance should be interpreted as **predictive contribution**, not causal influence.

In [ ]:
# Build readable names for the one-hot encoded feature vector.
feature_names = list(numerical_features)

# StringIndexerModel stages occur first in the fitted preprocessing pipeline.
for i, column in enumerate(categorical_features):
    indexer_model = preprocessing_model.stages[i]
    labels = list(indexer_model.labels)

    # OneHotEncoder uses dropLast=True by default.
    for label_value in labels[:-1]:
        feature_names.append(f"{column}={label_value}")

feature_importances = final_model.featureImportances.toArray()

print("Number of model features:", len(feature_importances))
print("Number of generated feature names:", len(feature_names))

In [ ]:
if len(feature_importances) != len(feature_names):
    raise ValueError(
        "Feature-name mapping does not match the model feature vector. "
        "Inspect the preprocessing pipeline before interpreting feature importance."
    )

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values("importance", ascending=False)

top_features = importance_df.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=top_features,
    x="importance",
    y="feature"
)
plt.title("Top 20 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

top_features

## 13. Business Interpretation

### Business Perspective

From a business analytics perspective, income classification can be viewed as a form of **customer or population segmentation**. The model identifies combinations of demographic, educational, employment and financial characteristics associated with different income groups.

Potential analytical applications include:

- workforce and labour-market analysis
- socioeconomic segmentation
- targeted research and survey analysis
- resource allocation studies
- understanding characteristics associated with different income segments

The model should **not** be interpreted as a tool for making high-stakes decisions about an individual's worth, employability or access to services.

### Management Insight

The value of the model is not only its predictive score. A business stakeholder can use the results to understand which variables are most informative, assess model trade-offs, and decide whether the model is sufficiently reliable for the intended analytical purpose.

## 14. Ethical Considerations and Limitations

Several limitations should be considered before applying the model outside this project:

1. **Historical data:** The Adult dataset reflects a historical population and may not represent current labour-market conditions.
2. **Geographic context:** The dataset is based on US census-related data and should not automatically be generalised to South Africa or another country.
3. **Class imbalance:** The `>50K` group is smaller than the `<=50K` group, so accuracy alone can be misleading.
4. **Demographic variables:** Variables such as race and sex may encode historical inequalities and can create fairness risks.
5. **Correlation is not causation:** Feature importance identifies variables useful for prediction; it does not show that a variable causes higher income.
6. **Model generalisation:** Performance on this test set does not guarantee performance on a new population.
7. **Responsible deployment:** Any real-world use would require additional validation, fairness testing, monitoring and governance.

These considerations are especially important for business decisions involving individuals.

## 15. Final Conclusion

This project developed an end-to-end income classification workflow using PySpark and the UCI Adult Census Income dataset.

The workflow covered:

- data acquisition and quality assessment
- missing-value treatment
- duplicate removal
- exploratory data analysis
- feature engineering
- categorical encoding
- scalable PySpark preprocessing
- Decision Tree modelling
- Random Forest modelling
- model comparison
- confusion-matrix analysis
- precision, recall, F1-score and AUC
- feature importance
- business interpretation
- ethical considerations

The final model should be judged using the complete set of evaluation metrics rather than accuracy alone. The project demonstrates how technical machine-learning methods can be combined with business reasoning to support responsible data-driven analysis.

## 16. Reproducibility

To reproduce the project:

1. Install Python and PySpark or open the notebook in Google Colab.
2. Obtain the UCI Adult dataset and place `adult.zip` in the working directory.
3. Run the notebook from top to bottom.
4. Review the generated EDA, model-comparison table, final confusion matrix and feature-importance chart.

Random seeds are fixed at `42` where applicable to improve reproducibility.